# Clustering Listwise DPO - four-arm run (1000 questions)

Runs the full comparison on an A100 runtime:

| arm | labels | ranking |
|---|---|---|
| `length` | gated (correct vs wrong by math-verify) | length |
| `gated_sm` | gated | bidirectional entailment, `nli-deberta-v3-small` |
| `gated_lg` | gated | bidirectional entailment, `nli-deberta-v3-large` |
| `ungated_lg` | binarized entailment only (PORT-style) | entailment |

All logic lives in the repo scripts (`run_entailment.sh`, `run_arms.sh`) - this notebook only wires up Colab: GPU check, Drive persistence, and the two script calls.
Generation is resume-safe: if the session disconnects, rerun the same cell and it continues where it left off.

In [ ]:
# -- Clone repo + install dependencies -----------------------------------------
!nvidia-smi
!git clone -b entailment-1000 https://github.com/txshah/clustering-listwise-dpo.git /content/clustering-listwise-dpo
%cd /content/clustering-listwise-dpo
!pip install -q "transformers==4.45.1" "trl==0.9.6" "peft==0.13.2" \
    "accelerate==1.13.0" "datasets==4.8.4" pyyaml math-verify sentencepiece
print("Done.")

In [ ]:
# -- Sanity check ---------------------------------------------------------------
# pip prints dependency warnings about jax/opencv/gradio etc. after the install:
# our pins need numpy<2 while Colab's preinstalled extras want numpy>=2.  Those
# packages are unused here, so the warnings are harmless IF this cell runs clean.
# On an ABI error ("numpy.dtype size changed"): Runtime > Restart session, rerun from the top.
import numpy, torch, transformers, trl, peft, datasets, math_verify
print("numpy", numpy.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| trl", trl.__version__)
torch.tensor([1.0]).numpy()
print("GPU:", torch.cuda.is_available())

In [ ]:
# -- Google Drive mount (persistence) + Hugging Face login ---------------------
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # paste your HF token when prompted (Mistral-7B needs an accepted license)

In [ ]:
# -- Config + Drive persistence -------------------------------------------------
# generation/traces and outputs are symlinked onto Drive, so raw traces, scored
# files, datasets, and trained adapters all survive a disconnect.
import os, shutil

N_QUESTIONS = 1000
N_SAMPLES   = 10    # traces per question; 20 roughly doubles usable records AND generation time
EVAL_BATCH  = 8     # questions per forward pass in eval (A100 handles 16)
EVAL_LIMIT  = ""    # "" = full 1319-question test split; e.g. 200 for a quick pass

REPO     = "/content/clustering-listwise-dpo"
WORK_DIR = f"/content/drive/MyDrive/clustering_listwise_dpo_{N_QUESTIONS}"

for sub in ["traces/raw", "traces/processed", "outputs"]:
    os.makedirs(f"{WORK_DIR}/{sub}", exist_ok=True)

def link(repo_path, drive_path):
    if os.path.islink(repo_path):
        return
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    os.symlink(drive_path, repo_path)

link(f"{REPO}/generation/traces", f"{WORK_DIR}/traces")
link(f"{REPO}/outputs",           f"{WORK_DIR}/outputs")
print(f"Persisting to {WORK_DIR}")

## Step 1: Generate the data (the long pole)

Prepare 1000 questions, sample traces at temperature 1, split correct/wrong with math-verify.
Roughly 5-7 h on an A100 at 10 samples/question.
Disconnected? Rerun this cell - already-generated questions are skipped.

In [ ]:
!N_QUESTIONS={N_QUESTIONS} N_SAMPLES={N_SAMPLES} bash run_entailment.sh prepare generate process

## Step 2: Score, build, train, eval (all four arms)

NLI scoring (both models), 4 datasets, 4 LoRA trainings, 5 evals (base + 4 arms), accuracy summary at the end.
Roughly 3-4 h on an A100 with full eval.
Reruns are cheap for every step except training; a subset also works, in order, e.g. `!bash run_arms.sh train eval`.

In [ ]:
!N_QUESTIONS={N_QUESTIONS} EVAL_BATCH={EVAL_BATCH} EVAL_LIMIT={EVAL_LIMIT} bash run_arms.sh

In [ ]:
# -- Copy result summaries to Drive + print them --------------------------------
import glob, json as _json, shutil as _shutil

print(f"{'model':28} {'accuracy':>9}")
for f in sorted(glob.glob("results_*.json")):
    _shutil.copy(f, WORK_DIR)
    with open(f) as fh:
        acc = _json.load(fh)["accuracy"]
    print(f"{f.replace('results_','').replace('.json',''):28} {acc:9.4f}")
print(f"\nCopied to {WORK_DIR}")

## Notes

- Eval output should say `Answer scorer: math-verify`; `regex fallback` means the install didn't take and numbers won't be comparable.
- Differences under ~2.5 accuracy points on the full test set are statistical ties - don't chase them.
- Trained adapters land in `outputs/arm_<name>_1000/` (on Drive via the symlink).